# 02. DCA-Trie v1: Static Semantic Filtering

**Purpose:** Implement and evaluate DCA-Trie v1 -- semantic filtering of KG paths before trie construction.

**Method:** Before building the KG-Trie, score each candidate path against the question using `sklearn.metrics.pairwise.cosine_similarity`. Keep only paths with cosine similarity >= tau.

**Reference:** Chapter 3, section 3.6 of the thesis. See `EXPLAINER.md` for background.

**Changes from GCR baseline:** Only the trie construction step changes. Everything else (model, beam search, evaluation) is identical -- isolating the effect.

In [ ]:
# @title 1. Install & Setup
import sys
import os
import torch
import json
from tqdm import tqdm
from datasets import load_dataset
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
gpu_arch = {75: "T4", 89: "L4", 80: "A100", 90: "H100"}
gpu_cc = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
has_a100 = "A100" in gpu_name or "H100" in gpu_name
print(f"GPU: {gpu_name}  |  VRAM: {gpu_mem:.1f} GB  |  Arch: {gpu_arch.get(gpu_cc, 'unknown')}")

!pip install -q transformers==4.44.2 "accelerate>=0.30.1" "datasets>=2.19.2" "marisa-trie>=1.2.0" "networkx>=3.0" "scikit-learn>=1.5.0" "tiktoken>=0.7.0" "sentencepiece>=0.2.0" "protobuf>=5.27.1"

flash_attn_installed = False
if has_a100:
    try:
        import flash_attn
        flash_attn_installed = True
        print(f"flash-attn already installed: {flash_attn.__version__}")
    except ImportError:
        print("Installing flash-attn (wheel)...")
        try:
            !pip install -q flash-attn --no-build-isolation
            import flash_attn
            flash_attn_installed = True
        except Exception:
            print("Wheel failed, trying source build...")
            try:
                !pip install -q flash-attn --no-build-isolation --no-cache-dir
                import flash_attn
                flash_attn_installed = True
            except Exception:
                print("flash-attn install failed, falling back to sdpa.")

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not os.path.exists("graph-constrained-reasoning"):
    !git clone https://github.com/Adjanour/graph-constrained-reasoning-dca
%cd graph-constrained-reasoning
sys.path.insert(0, '.')
from src.llms import get_registed_model

In [ ]:
# @title 2. Configuration
MODEL_PATH = "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct"
MODEL_NAME = "gcr"
# os.environ["HF_TOKEN"] = "hf_your_token_here"

DATA_PATH = "rmanluo"
DATASET = "RoG-webqsp"          # or "RoG-cwq"
SPLIT = "test"

# DCA-Trie v1 parameters
TAU = 0.25                       # admission threshold tau (tune on validation set!)
ENCODER_DEVICE = "cpu"           # keep on CPU to avoid GPU contention

INDEX_LEN = 2
K = 5
GEN_MODE = "group-beam"
PROMPT_MODE = "zero-shot"
MAX_NEW_TOKENS = 256
ATTN_IMPL = "flash_attention_2" if (has_a100 and flash_attn_installed) else "sdpa"

DRIVE_BASE = "/content/drive/MyDrive/gcr_results" if IN_COLAB else "results/GenPaths"
PREDICT_PATH = DRIVE_BASE
FORCE_RERUN = True

tag = f"DCAv1-tau{TAU}-cosine"
print(f"Model: {MODEL_PATH}")
print(f"Encoder device: {ENCODER_DEVICE}")
print(f"Threshold tau: {TAU}")
print(f"Dataset: {DATASET}")
print(f"Tag: {tag}")

In [ ]:
# @title 3. Load LLM
import argparse
from src.qa_prompt_builder import PathGenerationWithAnswerPromptBuilder

parser = argparse.ArgumentParser()
LLM = get_registed_model(MODEL_NAME)
LLM.add_args(parser)
args = parser.parse_args([])
args.model_path = MODEL_PATH
args.model_name = MODEL_NAME
args.k = K
args.generation_mode = GEN_MODE
args.attn_implementation = ATTN_IMPL
args.max_new_tokens = MAX_NEW_TOKENS
args.maximun_token = 4096

try:
    print("Loading LLM...")
    model = LLM(args)
    model.prepare_for_inference()
    print("LLM loaded.")
except Exception as e:
    print(f"LLM loading failed: {e}")
    raise

import src.utils as utils

In [ ]:
# @title 4. DCA-Trie v1: Build Semantically Filtered Trie

def build_dcav1_trie(question_dict, tokenizer, encoder, tau, index_path_length=2):
    """
    DCA-Trie v1: Static semantic filtering at construction time.

    Algorithm 1 from thesis:
      1. Compute query embedding u = E(query(q, epsilon))
      2. Enumerate paths P = BFS(G, Eq, L)
      3. For each p in P: keep if cos(u, E(str(p))) >= tau
      4. Build trie from filtered paths
    """
    question = question_dict["question"]
    q_embedding = encoder.encode(question, convert_to_numpy=True)

    if "paths" in question_dict:
        paths_list = question_dict["paths"]
    else:
        g = utils.build_graph(question_dict["graph"])
        paths_list = utils.dfs(g, question_dict["q_entity"], index_path_length)

    if len(paths_list) == 0:
        return None, [], []

    kept_paths = []
    kept_scores = []
    for p in paths_list:
        path_str = utils.path_to_string(p)
        p_embedding = encoder.encode(path_str, convert_to_numpy=True)
        score = cosine_similarity(q_embedding.reshape(1, -1), p_embedding.reshape(1, -1))[0][0]
        if score >= tau:
            kept_paths.append(p)
            kept_scores.append(score)

    if len(kept_paths) == 0:
        return None, [], []

    from src.trie import MarisaTrie
    paths_list_str = [utils.path_to_string(p) for p in kept_paths]
    tokenized = tokenizer(paths_list_str, padding=False, add_special_tokens=False).input_ids
    tokenized = [ids + [tokenizer.eos_token_id] for ids in tokenized]
    trie = MarisaTrie(tokenized, max_token_id=len(tokenizer) + 1)

    return trie, kept_paths, kept_scores

In [ ]:
# @title 5. Run DCA-Trie v1 Inference

dataset = load_dataset(f"{DATA_PATH}/{DATASET}", split=SPLIT)
print(f"Loaded {len(dataset)} questions")

model_short = MODEL_PATH.split("/")[-1]
postfix = f"{PROMPT_MODE}-{GEN_MODE}-k{K}-index_len{INDEX_LEN}-{tag}"
output_dir = os.path.join(PREDICT_PATH, DATASET, model_short, SPLIT, postfix)
os.makedirs(output_dir, exist_ok=True)
print(f"Output: {output_dir}")

def get_output(path, force):
    """Open predictions file for append or write. Resume from existing IDs."""
    if not os.path.exists(path) or force:
        return open(path, "w"), []
    with open(path, "r") as f:
        proc = [json.loads(l)["id"] for l in f]
    return open(path, "a"), proc

fout, processed = get_output(os.path.join(output_dir, "predictions.jsonl"), FORCE_RERUN)

total_paths_before = 0
total_paths_after = 0
total_empty = 0

try:
    for data in tqdm(dataset, desc="DCA-Trie v1"):
        qid = data["id"]
        if qid in processed:
            continue

        trie, kept_paths, scores = build_dcav1_trie(
            data, model.tokenizer, utils, TAU, INDEX_LEN
        )
        if trie is None:
            total_empty += 1
            continue

        total_paths_before += len(utils.dfs(
            utils.build_graph(data["graph"]), data["q_entity"], INDEX_LEN
        )) if "paths" not in data else len(data["paths"])
        total_paths_after += len(kept_paths)

        g = utils.build_graph(data["graph"])
        truth_paths = utils.get_truth_paths(data["q_entity"], data["a_entity"], g)
        ground_paths = [utils.path_to_string(p) for p in truth_paths]

        input_builder = PathGenerationWithAnswerPromptBuilder(model.tokenizer, PROMPT_MODE, index_path_length=INDEX_LEN)
        input_query, _, _ = input_builder.process_input(data)
        llm_input = model.prepare_model_prompt(input_query)

        start_ids = model.tokenizer.convert_tokens_to_ids(input_builder.PATH_START_TOKEN)
        end_ids = model.tokenizer.convert_tokens_to_ids(input_builder.PATH_END_TOKEN)

        prediction = model.generate_sentence(
            llm_input, trie,
            start_token_ids=start_ids,
            end_token_ids=end_ids,
            enable_constrained_by_default=False,
        )

        if prediction is None:
            continue

        fout.write(json.dumps({
            "id": qid,
            "question": data["question"],
            "prediction": prediction,
            "ground_truth": data["answer"],
            "ground_truth_paths": ground_paths,
            "input": input_query,
            "dca_tau": TAU,
            "dca_paths_before": int(total_paths_before / max(1, len(processed) + 1)),
            "dca_paths_after": int(total_paths_after / max(1, len(processed) + 1)),
        }) + "\n")
        fout.flush()
except Exception as e:
    print(f"Inference failed: {e}")
finally:
    fout.close()

print(f"\n=== Summary ===")
print(f"Questions with empty trie after filtering: {total_empty}")
print(f"Avg paths before filtering: {total_paths_before / max(1, len(dataset)):.1f}")
print(f"Avg paths after filtering (tau={TAU}): {total_paths_after / max(1, len(dataset)):.1f}")
print(f"Filter reduction: {(1 - total_paths_after/total_paths_before)*100:.1f}%")

from src.utils.qa_utils import eval_path_result_w_ans
print("\n=== Evaluation ===")
eval_path_result_w_ans(os.path.join(output_dir, "predictions.jsonl"))

In [ ]:
# @title 6. Threshold Validation (tune tau on a small held-out set)
print("Sweeping tau over WebQSP validation set (100 questions)...")

val_dataset = load_dataset(f"{DATA_PATH}/{DATASET}", split="test[:100]")

for tau_candidate in [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]:
    false_negatives = 0
    total = 0
    paths_before = 0
    paths_after = 0
    for data in val_dataset:
        g = utils.build_graph(data["graph"])
        truth_paths = utils.get_truth_paths(data["q_entity"], data["a_entity"], g)
        truth_strs = set(utils.path_to_string(p) for p in truth_paths)
        if len(truth_strs) == 0:
            continue
        total += 1

        trie, kept_paths, _ = build_dcav1_trie(data, model.tokenizer, utils, tau_candidate, INDEX_LEN)
        if trie is None:
            false_negatives += len(truth_strs)
            continue

        kept_strs = set(utils.path_to_string(p) for p in kept_paths)
        paths_before += len(utils.dfs(g, data["q_entity"], INDEX_LEN))
        paths_after += len(kept_paths)

        for t in truth_strs:
            if t not in kept_strs:
                false_negatives += 1

    fnr = false_negatives / max(1, total)
    reduction = (1 - paths_after / max(1, paths_before)) * 100 if paths_before > 0 else 0
    marker = " <<<" if fnr < 0.05 else ""
    print(f"  tau={tau_candidate:.2f}  |  FNR={fnr:.4f}  |  reduction={reduction:.1f}%{marker}")

print("\nPick the highest tau with FNR < 0.05 (Condition P from thesis).")

In [ ]:
# @title 7. Compare with GCR Baseline
print("""
=== Comparison ===

Metric               GCR         DCA-Trie v1 (yours)
Hits@1               92.6         ?
F1                   89.8         ?
Faithfulness        100%        100% (unchanged)
Avg trie size        ~5000       ? (should be lower)
SIR                  high        ? (should be lower)

To measure SIR, run notebook 04_SIR_Evaluation.ipynb
on the predictions.jsonl produced above.
""")